# 16. 보고서용 분석 (정의 · 피해 규모 · 예측 · 지원 알고리즘)

15번 노트북(지도) 결과를 이어받아 대회 요구사항에 필요한 표·그래프를 만든다.

**먼저 실행할 것**: 06, 07b, 08, 14, 15번 노트북 (15번 결과 `outputs/maps/grid_vulnerability_scores.csv`가 있어야 함)

| 셀 | 내용 | 주요 산출물 |
|---|---|---|
| 1 | 드라이브 연결, 한글 폰트 | — |
| 2 | 보고서 보조 그래프 3종 + 폭우×연령 확인 | `16_fig1_type.png`, `16_fig4_rain_dose.png`, `16_fig7_heat_age.png` |
| 3 | 기후 취약 상권 정의 · 호우 피해 규모 · 우선 지원 선정 알고리즘 | `16_vulnerable_grid.csv`, `16_grade_summary.csv`, `16_damage_scale.csv`, `16_priority_example.csv` |
| 4 | 폭염 피해 규모 (춘천 60대 이상 여가·외출형) | `16_heat_damage.csv` |
| 5 | 상권(격자)별 호우 예측 (07 예측 모델 → 격자) | `16_grid_rain_forecast.csv`, `16_grid_forecast_by_type.csv`, `16_fig_grid_forecast.png` |

그래프 글자에는 마이너스 기호로 `-`(일반 하이픈)를 쓴다. 나눔고딕에 `−` 글자가 없어 네모로 깨지기 때문이다.

## 1. 준비

In [ ]:
# 16-1. 드라이브 연결 + 한글 폰트
# 폰트를 처음 설치했다면: 런타임 > 세션 다시 시작 후 이 셀부터 다시 실행
from google.colab import drive
drive.mount("/content/drive")
!apt-get -qq -y install fonts-nanum > /dev/null


## 2. 보고서 보조 그래프 + 폭우 × 연령 확인
폭염과 같은 방식(12개 검정, FDR 보정)으로 폭우도 연령별로 확인한다.

In [ ]:
# ============================================================
# 16-2. 보고서용 보조 그래프 + 폭우 × 연령 확인
#   ① 상권 유형별 평균 예상 감소율   (outputs/maps/grid_vulnerability_scores.csv)
#   ④ 강수량 구간별 매출 변화         (outputs/tables/06_E_rain_dose.csv)
#   ⑦ 폭염: 연령 × 업종 유형          (outputs/tables/06_H4_age_type.csv)
#   + 폭우 × 연령 FDR 확인 (같은 06_H4 파일의 storm 열)
#   저장: outputs/figures/16_fig1_type.png, 16_fig4_rain_dose.png, 16_fig7_heat_age.png
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.patches import Patch
from statsmodels.stats.multitest import multipletests

nanum = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if Path(nanum).exists():
    fm.fontManager.addfont(nanum)
    plt.rcParams["font.family"] = fm.FontProperties(fname=nanum).get_name()
plt.rcParams["axes.unicode_minus"] = False

BASE = Path("/content/drive/MyDrive/bigcontest-2026")
TAB = BASE / "outputs" / "tables"
FIG = BASE / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
INK, SUB = "#1d2430", "#5b6472"
TCOL = {"낮·고령형": "#7b3294", "시내 생활형": "#1b9e77", "주말·여가형": "#e6ab02"}

# ------------------------------------------------------------
# 0. 폭우 × 연령 확인 (폭염과 같은 방식: 12개 검정 FDR)
# ------------------------------------------------------------
h4 = pd.read_csv(TAB / "06_H4_age_type.csv")
h4["heat_q"] = multipletests(h4["heat_p"], method="fdr_bh")[1]
h4["storm_q"] = multipletests(h4["storm_p"], method="fdr_bh")[1]
print("[폭우 × 연령 × 유형] 호우일 효과 (q<0.05면 유의)")
print(h4.assign(storm=(h4["storm"] * 100).round(1), storm_q=h4["storm_q"].round(4))
      [["region", "type", "age_grp", "storm", "storm_q"]].to_string(index=False))
print("→ 폭우 연령 유의 개수:", int((h4["storm_q"] < 0.05).sum()))

# ------------------------------------------------------------
# ① 상권 유형별 평균 예상 감소율
# ------------------------------------------------------------
g = pd.read_csv(BASE / "outputs" / "maps" / "grid_vulnerability_scores.csv")
t = g[g["region"] == "춘천"].groupby("type")[["heat_score", "rain_score"]].mean().loc[list(TCOL)]
fig, ax = plt.subplots(figsize=(8, 4.8)); fig.patch.set_facecolor("white")
y = np.arange(3)
ax.barh(y + .2, t["heat_score"] * 100, .38, color="#e4572e", label="폭염일")
ax.barh(y - .2, t["rain_score"] * 100, .38, color="#2171b5", label="호우일")
for i, (h, r) in enumerate(zip(t["heat_score"], t["rain_score"])):
    ax.text(h * 100 + .1, i + .2, f"-{h:.1%}", va="center", fontsize=11, color="#e4572e", fontweight="bold" if i == 0 else "normal")
    ax.text(r * 100 + .1, i - .2, f"-{r:.1%}", va="center", fontsize=11, color="#2171b5", fontweight="bold" if i == 0 else "normal")
ax.set_yticks(y); ax.set_yticklabels(list(TCOL), fontsize=12); ax.invert_yaxis()
for l in ax.get_yticklabels(): l.set_color(TCOL[l.get_text()]); l.set_fontweight("bold")
ax.set_xlim(0, 7.5); ax.set_xlabel("평균 예상 매출 감소율 (%)")
ax.legend(title="막대 색", frameon=False, loc="lower right")
for s in ["top", "right"]: ax.spines[s].set_visible(False)
ax.set_title("춘천 상권 유형별 예상 매출 감소율: 낮·고령형이 폭염·호우 모두 1위", fontsize=13, fontweight="bold", loc="left", color=INK)
fig.text(0.01, 0.01, "※ 격자별 예상 감소율(06 추정 효과 × 격자 구성)의 상권 유형별 평균. 강남은 유의한 감소가 없어 0%.", fontsize=9, color=SUB)
plt.tight_layout(rect=[0, 0.04, 1, 1]); fig.savefig(FIG / "16_fig1_type.png", dpi=300, facecolor="white"); plt.show()

# ------------------------------------------------------------
# ④ 강수량 구간별 매출 변화 (06 E)
# ------------------------------------------------------------
dose = pd.read_csv(TAB / "06_E_rain_dose.csv")
bins = ["무강수", "1~10mm", "10~30mm", "30mm+"]
LINES = {"춘천 야외레저": "#08306b", "춘천 카페·제과": "#41b6c4", "춘천 음식점": "#9aa1ab", "강남 야외레저": "#c4c8ce"}
fig, ax = plt.subplots(figsize=(9, 5.4)); fig.patch.set_facecolor("white")
for k, c in LINES.items():
    d = dose[dose["대상"] == k].set_index("구간").reindex(bins)
    v = d["변화"].values * 100; p = d["p"].fillna(1).values
    ax.plot(range(4), v, color=c, lw=2.5, zorder=2)
    for i in range(4):
        ax.scatter(i, v[i], s=70, color=c if p[i] < 0.05 else "white", edgecolor=c, lw=2, zorder=3)
    dy = {"춘천 음식점": 9, "강남 야외레저": -3, "춘천 카페·제과": -9}.get(k, 0)
    ax.annotate(f"{k} {v[-1]:+.1f}%", (3, v[-1]), xytext=(10, dy), textcoords="offset points",
                va="center", fontsize=10.5, color=c, fontweight="bold")
ax.axvspan(1.5, 2.5, color="#fff3b0", alpha=.6, zorder=0); ax.axvspan(2.5, 3.5, color="#fde0dd", alpha=.6, zorder=0)
ax.text(2, 12, "10mm 이상\n사전 알림", ha="center", fontsize=10.5, fontweight="bold", color="#8a6d00")
ax.text(3, 12, "30mm 이상\n피해 지원", ha="center", fontsize=10.5, fontweight="bold", color="#a3260f")
ax.axhline(0, color=INK, lw=.8); ax.set_xticks(range(4)); ax.set_xticklabels(bins); ax.set_xlim(-.2, 4.3)
ax.set_ylim(min(-62, ax.get_ylim()[0]), 20); ax.set_ylabel("비 안 온 날 대비 매출 변화 (%)")
for s in ["top", "right"]: ax.spines[s].set_visible(False)
ax.set_title("비가 많이 올수록 춘천 야외레저 매출이 줄어든다 (강수량 구간별)", fontsize=13, fontweight="bold", loc="left", color=INK)
fig.text(0.01, 0.01, "점: 채운 점 = 통계적으로 유의(p<0.05), 빈 점 = 유의하지 않음.  같은 주·요일 대비, 06 E 구간 분석.", fontsize=9, color=SUB)
plt.tight_layout(rect=[0, 0.04, 1, 1]); fig.savefig(FIG / "16_fig4_rain_dose.png", dpi=300, facecolor="white"); plt.show()

# ------------------------------------------------------------
# ⑦ 폭염: 연령 × 업종 유형 (06 H4)
# ------------------------------------------------------------
ages = ["10~30대", "40~50대", "60대 이상"]
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4), sharey=True); fig.patch.set_facecolor("white")
for ax, reg in zip(axes, ["춘천", "강남"]):
    for j, (typ, c) in enumerate([("여가·외출형", "#a3260f"), ("필수형", "#9aa1ab")]):
        d = h4[(h4["region"] == reg) & (h4["type"] == typ)].set_index("age_grp").reindex(ages)
        for i, a in enumerate(ages):
            v, q = d.loc[a, "heat"] * 100, d.loc[a, "heat_q"]; x = i + (j - .5) * .36
            ax.bar(x, v, .34, color=c if q < .05 else "white", edgecolor=c, lw=2)
            ax.text(x, v + (1 if v >= 0 else -1), f"{v:+.1f}%" + (" *" if q < .05 else ""), ha="center",
                    va="bottom" if v >= 0 else "top", fontsize=10.5, color=c, fontweight="bold" if q < .05 else "normal")
    ax.axhline(0, color=INK, lw=.8); ax.set_xticks(range(3)); ax.set_xticklabels(ages, fontsize=12)
    ax.set_title(reg, fontsize=14, fontweight="bold", loc="left", color=INK)
    for s in ["top", "right"]: ax.spines[s].set_visible(False)
axes[0].set_ylabel("폭염일 매출 변화 (%)"); axes[0].set_ylim(-27, 30)
fig.legend(handles=[Patch(facecolor="#a3260f", label="여가·외출형 (야외레저·숙박·스포츠·실내여가)"),
                    Patch(facecolor="#9aa1ab", label="필수형 (식료품·음식점·의료·카페)"),
                    Patch(facecolor="white", edgecolor=INK, label="빈 막대 = FDR 보정 후 유의하지 않음"),
                    Patch(facecolor=INK, label="채운 막대 * = 유의 (q<0.05)")],
           loc="lower center", ncol=2, frameon=False, fontsize=10.5)
fig.suptitle("폭염 피해는 춘천 60대 이상의 여가·외출 소비에서만 확인됐다 (12개 검정 중 1개)", fontsize=14, fontweight="bold", x=0.01, ha="left")
plt.tight_layout(rect=[0, 0.12, 1, 0.94]); fig.savefig(FIG / "16_fig7_heat_age.png", dpi=300, facecolor="white"); plt.show()
print("저장 완료:", FIG)


## 3. 기후 취약 상권 정의 · 호우 피해 규모 · 우선 지원 선정 알고리즘
- 정의: 예상 매출 감소율 5% 이상 = 취약 상권, 폭염·호우 둘 다 5% 이상 = 이중 취약
- 피해 규모: 08 추정 손실액 중 06에서 FDR 보정 후 유의한 감소 업종만
- 알고리즘: 예보(강수량, 최고기온) → 날씨 판정 → 취약 격자 → 행정동 우선순위 → 지원 내용

In [ ]:
# ============================================================
# 16-3. 기후 취약 상권 정의 · 피해 규모 · 우선 지원 선정 알고리즘
#
#   입력: outputs/maps/grid_vulnerability_scores.csv   (15번 노트북 결과)
#         outputs/tables/06_A_effects.csv               (06 업종 효과, FDR용)
#         outputs/tables/08_damage_by_industry.csv      (08 추정 손실액)
#         data/external/adm/adm_gangnam_chuncheon_2025.geojson
#   출력: outputs/tables/16_vulnerable_grid.csv      격자별 등급·취약 여부
#         outputs/tables/16_grade_summary.csv        등급별 칸 수
#         outputs/tables/16_damage_scale.csv         피해 규모 (FDR 유의 업종)
#         outputs/tables/16_priority_example.csv     알고리즘 실행 예시 결과
# ============================================================
import json
from pathlib import Path
import numpy as np
import pandas as pd
from matplotlib.path import Path as MPath
from pyproj import Transformer
from statsmodels.stats.multitest import multipletests

BASE = Path("/content/drive/MyDrive/bigcontest-2026")
TAB = BASE / "outputs" / "tables"
MAPS = BASE / "outputs" / "maps"
GEO = BASE / "data" / "external" / "adm" / "adm_gangnam_chuncheon_2025.geojson"

# ------------------------------------------------------------
# 1. 기후 취약 상권 정의
#   예상 매출 감소율(06 효과 × 격자 구성)로 4등급
#   취약 상권 = 폭염 또는 호우 예상 감소율 5% 이상
#   이중 취약 = 둘 다 5% 이상
# ------------------------------------------------------------
BINS = [-np.inf, 0.02, 0.05, 0.10, np.inf]
GRADES = ["일반", "관심", "취약", "고취약"]          # <2% / 2~5% / 5~10% / 10%+
VULN = 0.05

g = pd.read_csv(MAPS / "grid_vulnerability_scores.csv")
g["heat_grade"] = pd.cut(g["heat_score"], BINS, labels=GRADES, right=False)
g["rain_grade"] = pd.cut(g["rain_score"], BINS, labels=GRADES, right=False)
g["heat_vuln"] = g["heat_score"] >= VULN
g["rain_vuln"] = g["rain_score"] >= VULN
g["climate_vuln"] = g["heat_vuln"] | g["rain_vuln"]
g["double_vuln"] = g["heat_vuln"] & g["rain_vuln"]

# 행정동 붙이기
tf = Transformer.from_crs("EPSG:4326", "EPSG:5179", always_xy=True)
gj = json.load(open(GEO, encoding="utf-8"))
g["dong"] = None
for f in gj["features"]:
    name = f["properties"]["adm_nm"]
    reg = "강남" if "강남구" in name else ("춘천" if "춘천시" in name else None)
    if reg is None:
        continue
    m = (g["region"] == reg).values
    pts = g.loc[m, ["X_COORD", "Y_COORD"]].values
    geom = f["geometry"]
    hit = np.zeros(m.sum(), bool)
    for p in (geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]):
        r = np.array(p[0]); x, y = tf.transform(r[:, 0], r[:, 1])
        hit |= MPath(np.c_[x, y]).contains_points(pts)
    idx = g.index[m][hit]
    g.loc[idx, "dong"] = name.split()[-1]

g.to_csv(TAB / "16_vulnerable_grid.csv", index=False, encoding="utf-8-sig")

summary = pd.concat({
    "폭염": g.groupby(["region", "heat_grade"], observed=False).size().unstack(fill_value=0),
    "호우": g.groupby(["region", "rain_grade"], observed=False).size().unstack(fill_value=0),
}).reindex(columns=GRADES)
summary.to_csv(TAB / "16_grade_summary.csv", encoding="utf-8-sig")
print("[1] 등급별 상권 칸 수 (일반 <2% / 관심 2~5% / 취약 5~10% / 고취약 10%+)")
print(summary)

ch = g[g["region"] == "춘천"]
dv = ch[ch["double_vuln"]]
print(f"\n기후 취약 상권(둘 중 하나 5% 이상): 춘천 {int(ch['climate_vuln'].sum())}칸 / 강남 {int(g[g['region']=='강남']['climate_vuln'].sum())}칸")
print(f"이중 취약: {len(dv)}칸, 그중 낮·고령형 {(dv['type'] == '낮·고령형').mean():.0%}")

# ------------------------------------------------------------
# 2. 피해 규모: 08 추정 손실액 중 06에서 FDR 보정 후 유의한 업종만
#   (08 파일의 '유의' 열은 보정 전 p<0.05라서 06 기준으로 다시 거름)
# ------------------------------------------------------------
eff = pd.read_csv(TAB / "06_A_effects.csv")
n = len(eff)
q = multipletests(pd.concat([eff["heat_p"], eff["rainy_p"], eff["storm_p"]]).values, method="fdr_bh")[1]
eff["heat_q"], eff["storm_q"] = q[:n], q[2 * n:]

dmg = pd.read_csv(TAB / "08_damage_by_industry.csv").merge(
    eff[["region", "ind_grp", "heat", "heat_q", "storm", "storm_q"]], on=["region", "ind_grp"])
rows = []
for w, loss, qcol, ecol in [("호우", "호우손실", "storm_q", "storm"), ("폭염", "폭염손실", "heat_q", "heat")]:
    d = dmg[(dmg[qcol] < 0.05) & (dmg[ecol] < 0)]          # 보정 후 유의한 '감소'만
    for _, r in d.iterrows():
        rows.append({"날씨": w, "지역": r["region"], "업종": r["ind_grp"],
                     "추정 손실(억 원)": round(r[loss] / 1e8, 1),
                     "여름 매출 대비(%)": round(r[loss] / r["여름매출"] * 100, 1)})
scale = pd.DataFrame(rows)
scale.to_csv(TAB / "16_damage_scale.csv", index=False, encoding="utf-8-sig")
print("\n[2] 피해 규모 (2025년 7~9월, 신한카드 결제 기준 추정)")
print(scale.to_string(index=False) if len(scale) else "보정 후 유의한 감소 업종 없음")
if len(scale):
    print(scale.groupby(["날씨", "지역"])["추정 손실(억 원)"].sum())

# ------------------------------------------------------------
# 3. 우선 지원 대상 선정 알고리즘
#   입력: 지역별 예보 (강수량 mm, 최고기온 ℃)
#   ① 날씨 판정  ② 해당 날씨 점수 선택  ③ 등급·이중 취약 정렬  ④ 지원 매칭
# ------------------------------------------------------------
ACTION = {
    "호우 경보": "피해 지원 (예약 변경 수수료 완화, 대체 프로그램 홍보)",
    "호우 주의": "사전 알림 (배달·포장 연계, 영업시간 조정 안내)",
    "폭염": "냉방·그늘막·무더위 쉼터, 어르신 외출 시간대 할인",
}

def judge(rain_mm, tmax):
    out = []
    if rain_mm >= 30: out.append("호우 경보")
    elif rain_mm >= 10: out.append("호우 주의")
    if tmax >= 33: out.append("폭염")
    return out

def select_priority(forecast, top=10):
    """forecast = {"춘천": {"rain_mm": 45, "tmax": 29}, "강남": {...}}"""
    res = []
    for reg, fc in forecast.items():
        for level in judge(fc["rain_mm"], fc["tmax"]):
            col = "heat_score" if level == "폭염" else "rain_score"
            d = g[(g["region"] == reg) & (g[col] >= VULN)].copy()
            if d.empty:
                continue
            d["날씨"] = level
            d["예상 감소율(%)"] = (d[col] * 100).round(1)
            d["등급"] = pd.cut(d[col], BINS, labels=GRADES, right=False).astype(str)
            res.append(d)
    if not res:
        return pd.DataFrame()
    r = pd.concat(res)
    # 동 × 날씨 단위로 묶어 우선순위: 고취약 칸 수 → 이중 취약 칸 수 → 평균 예상 감소율
    out = (r.groupby(["region", "dong", "날씨"])
             .agg(고취약_칸=("등급", lambda s: (s == "고취약").sum()),
                  이중취약_칸=("double_vuln", "sum"),
                  취약_칸=("등급", "size"),
                  평균_예상감소율=("예상 감소율(%)", "mean"),
                  골프장_레고랜드=("has_golf", lambda s: int(s.sum() + r.loc[s.index, "has_lego"].sum())),
                  주요_상권유형=("type", lambda s: s.mode().iat[0]))
             .reset_index()
             .sort_values(["고취약_칸", "이중취약_칸", "평균_예상감소율"], ascending=False))
    out["평균_예상감소율"] = out["평균_예상감소율"].round(1)
    out["지원"] = out["날씨"].map(ACTION)
    out.insert(0, "순위", range(1, len(out) + 1))
    return out.head(top)

example = select_priority({"춘천": {"rain_mm": 45, "tmax": 34}, "강남": {"rain_mm": 45, "tmax": 34}})
example.to_csv(TAB / "16_priority_example.csv", index=False, encoding="utf-8-sig")
print("\n[3] 알고리즘 실행 예시: 강남·춘천 모두 45mm 호우 + 34℃ 폭염 예보")
print(example.to_string(index=False))
print("\n강남은 예상 감소율 5% 이상인 칸이 없어 목록에 나오지 않음 (유의한 감소 없음)")


## 4. 폭염 피해 규모
06 H4와 같은 데이터·같은 회귀식으로 효과를 다시 추정하고 08과 같은 공식으로 손실액을 구한다.
`손실 = Σ(폭염일 실제 매출) × (exp(-β) - 1)` = 폭염이 아니었다면 벌었을 매출 - 실제 매출

In [ ]:
# ============================================================
# 16-4. 폭염 피해 규모: 춘천 60대 이상 여가·외출형
#   06 H1·H4와 같은 데이터·같은 회귀식으로 폭염 효과를 다시 추정하고
#   08과 같은 공식으로 폭염일 손실액을 계산한다.
#     손실 = Σ(폭염일 실제 매출) × (exp(−β) − 1)
#     → "폭염이 아니었다면 벌었을 매출 − 실제 매출"
#   입력: data/processed/analysis_visitor.csv, docs/industry_mapping.csv,
#         data/raw/card/신한카드_빅콘테스트2026_데이터2.txt (읽기만 함)
#   출력: outputs/tables/16_heat_damage.csv
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

BASE = Path("/content/drive/MyDrive/bigcontest-2026")
OUT = BASE / "data" / "processed"
RAW_CARD = BASE / "data" / "raw" / "card"
DOCS = BASE / "docs"
TAB = BASE / "outputs" / "tables"

FORMULA = "lsales ~ heat + rainy + heavy + C(week) + C(dow)"     # 06과 동일

def add_weather_flags(df, heat_c=33, heavy_mm=30):
    df = df.copy()
    df["heat"] = (df["tmax"] >= heat_c).astype(int)
    df["rainy"] = (df["rain"] >= 1).astype(int)
    df["heavy"] = (df["rain"] >= heavy_mm).astype(int)
    df["lsales"] = np.log(df["sales"])
    return df

# ---- 1. 날짜별 날씨 (06과 동일: 7~9월, 광복절 제외) ----
av = pd.read_csv(OUT / "analysis_visitor.csv", parse_dates=["date"])
av = av[(av["summer"] == 1) & (av["holiday"] == 0)]
DAY_COLS = ["date", "region", "dow", "week", "tmax", "feels_max", "rain", "heat_streak"]
days = av[DAY_COLS].drop_duplicates()

# ---- 2. 연령별 카드 매출 (06 H1과 동일) ----
mapping = pd.read_csv(DOCS / "industry_mapping.csv")
keep = mapping[~mapping["ind_grp"].str.startswith("제외")]
AGE_MAP = {"10대": "10~30대", "20대": "10~30대", "30대": "10~30대",
           "40대": "40~50대", "50대": "40~50대",
           "60대": "60대 이상", "70대": "60대 이상", "80대": "60대 이상", "90대": "60대 이상"}
a = pd.read_csv(RAW_CARD / "신한카드_빅콘테스트2026_데이터2.txt", sep="\t", encoding="cp949",
                dtype=str, usecols=["TA_YMD", "MCT_SGG_CD", "MCT_RY_CD", "AGE_CCD", "TS_AT"])
a = a.rename(columns={"TA_YMD": "date", "MCT_SGG_CD": "region", "MCT_RY_CD": "industry",
                      "AGE_CCD": "age", "TS_AT": "sales"})
a["date"] = pd.to_datetime(a["date"], format="%Y%m%d")
a["region"] = a["region"].map({"서울 강남구": "강남", "강원 춘천시": "춘천"})
a["sales"] = pd.to_numeric(a["sales"])
a["age_grp"] = a["age"].str.replace(" ", "").map(AGE_MAP)
a = a.merge(keep, on="industry")

# ---- 3. 06 H4와 같은 유형 묶음 (유흥업소 제외) ----
LEISURE = ["야외레저", "숙박", "스포츠", "실내여가"]
ESSENTIAL = ["식료품소매", "음식점", "의료", "카페·제과"]
a2 = a[a["industry"] != "유흥업소"].copy()
a2["type"] = np.select([a2["ind_grp"].isin(LEISURE), a2["ind_grp"].isin(ESSENTIAL)],
                       ["여가·외출형", "필수형"], default="")
a2 = a2[a2["type"] != ""]
t = a2.groupby(["date", "region", "type", "age_grp"])["sales"].sum().reset_index()
t = add_weather_flags(t.merge(days, on=["date", "region"]))   # 여름 평일·주말(광복절 제외)만 남음

# ---- 4. 12개 조합 모두 회귀 → FDR 보정 → 유의한 '감소'만 손실액 계산 ----
rows = []
for (reg, typ, age), d in t.groupby(["region", "type", "age_grp"]):
    m = smf.ols(FORMULA, data=d).fit(cov_type="HC1")
    b = m.params["heat"]
    hd = d[d["heat"] == 1]
    loss = (hd["sales"] * (np.exp(-b) - 1)).sum()          # 08과 같은 공식 (양수 = 손실)
    rows.append({"지역": reg, "유형": typ, "연령": age,
                 "폭염 효과(%)": (np.exp(b) - 1) * 100, "p": m.pvalues["heat"],
                 "폭염일 수": int(d.drop_duplicates("date")["heat"].sum()),
                 "폭염일 실제 매출(억 원)": hd["sales"].sum() / 1e8,
                 "여름 매출(억 원)": d["sales"].sum() / 1e8,
                 "추정 손실(억 원)": loss / 1e8})
res = pd.DataFrame(rows)
res["q"] = multipletests(res["p"], method="fdr_bh")[1]
res["여름 매출 대비(%)"] = res["추정 손실(억 원)"] / res["여름 매출(억 원)"] * 100

show = res.copy()
for c in ["폭염 효과(%)", "폭염일 실제 매출(억 원)", "여름 매출(억 원)", "추정 손실(억 원)", "여름 매출 대비(%)"]:
    show[c] = show[c].round(2)
show["p"], show["q"] = show["p"].round(4), show["q"].round(4)
print("[폭염 × 연령 × 유형] 12개 조합 (06 H4 재현 확인용: 춘천 여가·외출형 60대 이상이 -18.9%, q≈0.0003이면 정상)")
print(show.to_string(index=False))

sig = res[(res["q"] < 0.05) & (res["폭염 효과(%)"] < 0)]
sig.round(4).to_csv(TAB / "16_heat_damage.csv", index=False, encoding="utf-8-sig")
print("\n[폭염 피해 규모] FDR 보정 후 유의한 감소만")
if len(sig):
    for _, r in sig.iterrows():
        print(f"  {r['지역']} {r['유형']} {r['연령']}: 폭염일 {r['폭염일 수']}일, "
              f"추정 손실 {r['추정 손실(억 원)']:.2f}억 원 (이 집단 여름 매출의 {r['여름 매출 대비(%)']:.2f}%)")
else:
    print("  보정 후 유의한 감소 없음")
print("※ 신한카드 결제 기준 추정액이라 실제 지역 전체 매출 손실보다 작다.")


## 5. 상권(격자)별 호우 예측
07b(07 LightGBM 시나리오)의 지역×업종 예상 변화를 격자 가게 구성에 붙여 격자별 예상 변화를 만들고, 06 기반 점수와 같은 격자를 위험하다고 보는지 교차 확인한다.

In [ ]:
# ============================================================
# 16-5. 상권(격자)별 호우 예측: 07b 예측 모델 시나리오 → 50m 격자
#   07b: 07 LightGBM(③ 날씨 포함)으로 "비 안 온 여름날에 강수량만 10/30/45mm로 바꿨을 때"
#        지역 × 업종별 예상 매출 변화(%)를 계산한 결과 (07_scenario.csv)
#   여기서: 격자 안 가게들의 업종 구성에 07b 값을 붙여 격자별 예상 변화(%)를 만든다
#     격자 예상 변화 = Σ(가게별 업종 예상 변화) ÷ 격자 전체 가게 수   (15번 호우 점수와 같은 방식)
#     골프장·레고랜드 칸 = 춘천 야외레저 값
#   15번(06 회귀 효과)과 다른 점: 06은 유의한 업종만, 07b는 모델이 예측한 모든 업종 값 사용
#                               → 강남도 값이 나온다 (작지만 0이 아님)
#   입력: data/processed/grid_types.csv, stores_grid.csv, outputs/tables/07_scenario.csv,
#         outputs/tables/16_vulnerable_grid.csv (16-3 결과, 행정동·06 점수)
#   출력: outputs/tables/16_grid_rain_forecast.csv, 16_grid_forecast_by_type.csv
#         outputs/figures/16_fig_grid_forecast.png
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

nanum = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if Path(nanum).exists():
    fm.fontManager.addfont(nanum)
    plt.rcParams["font.family"] = fm.FontProperties(fname=nanum).get_name()
plt.rcParams["axes.unicode_minus"] = False

BASE = Path("/content/drive/MyDrive/bigcontest-2026")
DATA = BASE / "data" / "processed"
TAB = BASE / "outputs" / "tables"
FIG = BASE / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
MM = [10, 30, 45]

# ---- 1. 데이터 ----
grid = pd.read_csv(DATA / "grid_types.csv")
stores = pd.read_csv(DATA / "stores_grid.csv", low_memory=False)
scen = pd.read_csv(TAB / "07_scenario.csv")                     # 지역, 업종, 10mm, 30mm, 45mm

# 06에서 해석 제외한 춘천 대형유통(기타유통 주도)은 07b 그래프와 같이 빼고 0으로 둔다
scen.loc[(scen["지역"] == "춘천") & (scen["업종"] == "대형유통"), [f"{m}mm" for m in MM]] = 0.0

grid["X_JOIN"], grid["Y_JOIN"] = grid["X_COORD"].round(2), grid["Y_COORD"].round(2)
stores["X_JOIN"], stores["Y_JOIN"] = stores["grid_x"].round(2), stores["grid_y"].round(2)
in_grid = stores["in_grid"].astype(str).str.strip().str.lower().isin(["true", "1"])
stores = stores[in_grid].dropna(subset=["X_JOIN", "Y_JOIN"]).copy()
stores["ind_grp"] = stores["ind_grp"].astype(str).str.strip()

# ---- 2. 가게마다 업종 예상 변화 붙이기 → 격자 평균 ----
stores = stores.merge(scen.rename(columns={"지역": "region", "업종": "ind_grp"}),
                      on=["region", "ind_grp"], how="left")
for m in MM:
    stores[f"{m}mm"] = stores[f"{m}mm"].fillna(0)               # 07b에 없는 업종 = 0
s = stores.groupby(["region", "X_JOIN", "Y_JOIN"])[[f"{m}mm" for m in MM]].sum().reset_index()
grid = grid.merge(s, on=["region", "X_JOIN", "Y_JOIN"], how="left")
special = ((grid["has_golf"] > 0) | (grid["has_lego"] > 0)) & (grid["region"] == "춘천")
leisure = scen[(scen["지역"] == "춘천") & (scen["업종"] == "야외레저")].iloc[0]
for m in MM:
    c = f"pred_{m}mm"
    grid[c] = grid[f"{m}mm"].fillna(0) / grid["n_stores"].clip(lower=1)
    grid.loc[special, c] = leisure[f"{m}mm"]
grid = grid.drop(columns=[f"{m}mm" for m in MM])
print("[격자별 예상 매출 변화(%)] 지역별 평균 / 가장 큰 감소")
print(grid.groupby("region")[[f"pred_{m}mm" for m in MM]].agg(["mean", "min"]).round(2).to_string())

# ---- 3. 행정동·06 점수 붙이기 (16-3 결과) ----
v = pd.read_csv(TAB / "16_vulnerable_grid.csv")
v["X_JOIN"], v["Y_JOIN"] = v["X_COORD"].round(2), v["Y_COORD"].round(2)
grid = grid.merge(v[["region", "X_JOIN", "Y_JOIN", "dong", "rain_score"]], on=["region", "X_JOIN", "Y_JOIN"], how="left")

# 두 방법(06 회귀 효과 vs 07b 예측 모델)이 같은 격자를 위험하다고 보는지 확인 (춘천)
ch = grid[grid["region"] == "춘천"]
r = ch["rain_score"].corr(-ch["pred_45mm"])
hi06 = ch["rain_score"] >= 0.10            # 06 기준 고취약 (10% 이상)
hi07 = ch["pred_45mm"] <= -10              # 07b 기준 45mm에서 10% 이상 감소
print(f"\n[교차 확인] 춘천 격자: 06 호우 점수와 07b 45mm 예측의 상관 = {r:.2f}")
print(f"  06 고취약 {hi06.sum()}칸 중 07b도 10% 이상 감소로 예측한 칸 = {(hi06 & hi07).sum()}칸")
print(f"  07b 10% 이상 감소 {hi07.sum()}칸 중 06 고취약 칸 = {(hi06 & hi07).sum()}칸")

print("\n[45mm 예보 시 예상 감소 크기별 격자 수]")
tail = pd.DataFrame({f"{k}% 이상 감소": grid.groupby("region")["pred_45mm"].apply(lambda x, k=k: int((x <= -k).sum()))
                     for k in [5, 10, 20]})
tail["전체 칸"] = grid.groupby("region").size()
print(tail.to_string())
print("※ 07b 결과 해석과 같이 ±5% 안팎의 작은 값은 모델 설정에 따라 흔들리므로 해석하지 않는다.")
gn10 = grid[(grid["region"] == "강남") & (grid["pred_45mm"] <= -10)]
if len(gn10):
    comp = stores.merge(gn10[["region", "X_JOIN", "Y_JOIN"]], on=["region", "X_JOIN", "Y_JOIN"])["ind_grp"].value_counts()
    print(f"  강남 10% 이상 감소 {len(gn10)}칸: 칸당 가게 수 {sorted(gn10['n_stores'].astype(int).tolist())}, 가게 업종 {comp.to_dict()}")
    print("  (06 회귀에서 강남은 FDR 보정 후 유의한 호우 감소 업종이 없음)")

out_cols = ["region", "X_COORD", "Y_COORD", "dong", "type", "n_stores", "has_golf", "has_lego"] + [f"pred_{m}mm" for m in MM]
grid[out_cols].round(3).to_csv(TAB / "16_grid_rain_forecast.csv", index=False, encoding="utf-8-sig")

# ---- 4. 상권 유형별 요약 ----
by_type = grid.groupby(["region", "type"]).agg(칸수=("n_stores", "size"),
                                              **{f"{m}mm": (f"pred_{m}mm", "mean") for m in MM}).round(2)
by_type.to_csv(TAB / "16_grid_forecast_by_type.csv", encoding="utf-8-sig")
print("\n[상권 유형별 평균 예상 매출 변화(%)]")
print(by_type.to_string())

print("\n[45mm 예보 시 예상 감소가 큰 행정동 TOP 8] (격자 5칸 이상인 동만)")
dong = (grid.dropna(subset=["dong"]).groupby(["region", "dong"])
        .agg(칸수=("pred_45mm", "size"), 평균_예상변화=("pred_45mm", "mean"), 최저=("pred_45mm", "min")))
dong = dong[dong["칸수"] >= 5].sort_values("평균_예상변화").round(2)
print(dong.head(8).to_string())

# ---- 5. 예보 강수량 → 격자 예상 변화 (예측 함수) ----
def predict_grid(region, rain_mm):
    """예보 강수량(mm)을 넣으면 그 지역 격자별 예상 매출 변화(%)를 돌려준다.
    07b 계산값(0/10/30/45mm) 사이는 직선으로 이어서 읽고, 45mm 초과는 45mm 값 사용(밖으로 늘리지 않음)."""
    xs = [0] + MM
    d = grid[grid["region"] == region].copy()
    vals = np.c_[np.zeros(len(d)), d[[f"pred_{m}mm" for m in MM]].values]
    x = min(max(rain_mm, 0), 45)
    d["예상 변화(%)"] = [np.interp(x, xs, row) for row in vals]
    return d.sort_values("예상 변화(%)")[["region", "dong", "type", "n_stores", "예상 변화(%)"]]

ex = predict_grid("춘천", 20)
print("\n[예시] 춘천 20mm 예보 → 행정동별 예상 변화 (격자 5칸 이상, 큰 감소 순 5개)")
ex = ex.dropna(subset=["dong"]).groupby("dong")["예상 변화(%)"].agg(칸수="size", 평균="mean", 최저="min")
print(ex[ex["칸수"] >= 5].sort_values("평균").head(5).round(2).to_string())

# ---- 6. 그래프: 상권 유형 × 강수량 ----
TYPES = {"강남": ["평일·업무형", "번화가형", "주거·고령형", "주말·여가형"],
         "춘천": ["낮·고령형", "시내 생활형", "주말·여가형"]}
COL = {10: "#c6dbef", 30: "#6baed6", 45: "#08519c"}
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True); fig.patch.set_facecolor("white")
for ax, reg in zip(axes, ["춘천", "강남"]):
    types = [t for t in TYPES[reg] if (reg, t) in by_type.index]
    x = np.arange(len(types))
    for j, m in enumerate(MM):
        vals = [by_type.loc[(reg, t), f"{m}mm"] for t in types]
        ax.bar(x + (j - 1) * .26, vals, .25, color=COL[m], label=f"{m}mm 예보")
        for xi, v_ in zip(x + (j - 1) * .26, vals):
            ax.text(xi, v_ - .15, f"{v_:.1f}", ha="center", va="top", fontsize=9)
    ax.axhline(0, color="#1d2430", lw=.8)
    ax.set_xticks(x); ax.set_xticklabels(types, fontsize=11)
    ax.set_title(reg, fontsize=14, fontweight="bold", loc="left")
    for sp in ["top", "right"]: ax.spines[sp].set_visible(False)
axes[0].set_ylabel("비 안 온 날 대비 예상 매출 변화 (%)")
lo = min(by_type[[f"{m}mm" for m in MM]].min().min() * 1.25, -1)
axes[0].set_ylim(lo, 1)
axes[1].legend(title="막대 색 = 예보 강수량", frameon=False, loc="lower right")
fig.suptitle("상권 유형별 호우 예측 (07 예측 모델): 춘천 낮·고령형이 가장 크게 줄어든다",
             fontsize=13, fontweight="bold", x=0.01, ha="left")
fig.text(0.01, 0.01, "※ 07 예측 모델(LightGBM) 시나리오(07b)의 업종별 예상 변화를 격자 가게 구성으로 평균. "
         "막대 아래 숫자 = 유형 평균(%). ±5% 안팎은 모델 오차 범위라 해석하지 않음.", fontsize=9, color="#5b6472")
plt.tight_layout(rect=[0, 0.04, 1, 0.93]); fig.savefig(FIG / "16_fig_grid_forecast.png", dpi=300, facecolor="white"); plt.show()
